<img src="https://raw.githubusercontent.com/mctools/ncrystal-logo/main/png/logo-compact/ncrystal-logo-compact-64h.png" alt="NCrystal" width="158">

# Example of how to create ENDF library with Extinction

In [ ]:
#Install conda on Google Colab. This restarts the kernel, so the
#notebook will say that it crashed. This is expected!
%pip -q install "condacolab<0.2"
import condacolab
condacolab.install_miniforge()

In [ ]:
#Fix the wrapper of the Python executable from condacolab 0.1:
import pathlib as _pl, sys as _sys
_w = _pl.Path(_sys.executable.removesuffix(".real"))
if _w.is_file() and " -x $@" in _w.read_text():
    _w.write_text(_w.read_text().replace(" -x $@", ' "$@"'))
#Install software on Google Colab:
!mamba install -y -q --override-channels -c conda-forge matplotlib ipympl ncrystal-all requests openmc njoy2016 openssl
%pip -q install ncrystal-notebook-utilities "ncrystal-plugin-CrysExtn @ git+https://github.com/XuShuqi7/ncplugin-CrysExtn"
#Enable widgets on Google Colab:
from google.colab import output as _google_colab_output
_google_colab_output.enable_custom_widget_manager()
#Matplotlib plots:
%matplotlib inline
import matplotlib
matplotlib.rcParams.update({"figure.autolayout": True})
#Avoid warnings later on Google Colab (cf. mctools/ncrystal#192):
import gemmi
import spglib
#Import NCrystal:
import NCrystal as NC
assert NC.version_num >= 4004006, "NCrystal version 4.4.6 or later is needed"
NC.test() #< quick test that the installation works
#Check that NCrystal finds the plugin:
assert 'CrysExtn' in [ p[0] for p in NC.browsePlugins() ], 'NCrystal plugin CrysExtn not found'

<a id="endfextnplugin-intro"></a>

## Introduction
This example shows how to use the extinction plugin CrysExtn [1] together with the ncmat2endf script to generate an ENDF library. Extinction is included as a modification to the coherent elastic cross-section and is included via a standalone NCrystal plugin.

The coherent scattering cross-section is given by,
\begin{equation}
\sigma_{\textrm{coh}}^{\textrm{el}}(\lambda)=\dfrac{\lambda^{2}}{2V}\sum_{hkl}^{\lambda\leq 2d_{hkl}}d_{hkl}|F_{hkl}|^{2}E_{hkl}(\lambda,d_{hkl}),
\end{equation}
where $E_{hkl}(\lambda,d_{hkl})$ is the extinction factor. Several different models are available in the plugin, however this example will focus on primary extinction using the Sabine model [2,3]. More information on available models can be found in [1] in the files `README.md` and `src/NCPhysicsModel.cc`.

* [1] https://github.com/XuShuqi7/ncplugin-CrysExtn
* [2] Sabine, T. (1985). Aust. J. Phys. 38(3), 507-518
* [3] Sabine, T. (2006). International Tables for Crystallography (2006). Vol. C. ch. 6.4, pp. 609-616

This notebook needs the CrysExtn plugin, which is installed together with its other requirements (see the top of the notebook). It can be found, together with other NCrystal plugins, at https://github.com/mctools/ncrystal/wiki/CuratedPlugins.

<a id="endfextnplugin-usage"></a>

## Using the plugin
Let's create a file and set the parameters of the model.

This is done by taking an existing file and adding the relevant section for the new physics through using `c.set_custom_section_data()`.
The name of the plugin is `CRYSEXTN` followed by the data used by the model.
The `CRYSEXTN` plugin currently supports both the uncorrelated and correlated model of Sabine, however with only primary extinction active, both models are the same. This example will use the correlated model. Primary extinction is controlled by the crystallite size parameter $l$, while secondary extinction is controlled by the grain size parameter $L$ and the mosacity parameter $g$ [3]. The parameters $g$ and $L$ are set to zero for only primary extinction. $l$ and $L$ have units of Ångströms and $g$ has units of 1/rad.

The data is added to the section called `@CUSTOM_CRYSEXTN`. Sabine_corr indicates the Sabine correlated model. The first parameter is $l$, the second is $g$ and the third is $L$.

In [ ]:
#Base our material from Beryllium from the NCrystal std data library:
c = NC.NCMATComposer('stdlib::Be_sg194.ncmat')
#Enable primary extinction, with 5 micron crystal domains (units are Angstrom):
c.set_custom_section_data("CRYSEXTN", " Sabine_corr 5e+4  0  0 ")
c.register_as('Be_sg194-CrysExtn.ncmat')#register as virtual file, use .write(..) to instead create the file
print( c() )# print the NCMAT data

Compare against the ideal polycrystalline Be cross-section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wl = np.linspace(0.2, 6, 10000) #Wavelengths, Aa

#Define NCrystal cfg-strings:
cfg_std = 'stdlib::Be_sg194.ncmat;temp=293.6K'
cfg_extn = 'Be_sg194-CrysExtn.ncmat;temp=293.6K' #<-- the data we registered above

#Load as NCrystal objects:
mat_std = NC.load(cfg_std)
mat_extn = NC.load(cfg_extn)

#Plot cross sections:
plt.figure()
plt.plot(NC.wl2ekin(wl), mat_extn.xsect(wl=wl), 'r-', label='293.6 K with extinction')
plt.plot(NC.wl2ekin(wl), mat_std.xsect(wl=wl), 'b-', label='293.6 K')
plt.legend()
plt.xscale('log')
plt.xlabel('Energy [eV]')
plt.ylabel('Micro scattering cross section [b/atom]')
plt.show()

Now we will setup up the parameters for the ncmat2endf script.

If we try to do this blindly, we will get an error:

* `ERROR: Material configuration indicates scattering processes which has not been vetted for conversion to the ENDF format: "CrysExtnModel"`

This is because the code in `ncmat2endf` only knows about physics models from NCrystal itself, not those made available via a plugin. To get around this, we currently have to use a somewhat nasty hack, adding the `CrysExtnModel` to the list of models that `ncmat2endf` will accept (NOTE: This hack will be formalised in a future NCrystal release!):

In [ ]:
from NCrystal import _ncmat2endf_impl
_ncmat2endf_impl.allowed_scat_proc_names.add('CrysExtnModel')

Now we can finally start on exporting the material with extinction to ENDF. First we will fill out the relevant ENDF metadata.

In [ ]:
from NCrystal import ncmat2endf
m = ncmat2endf.EndfMetaData()
m.set_value("alab","MyLabName")
m.set_value("libname","NuclearDataLibraryName")
m.set_value("auth","AuthorNames")
m.set_value("nlib",2)
m.set_value("matnum",{"Be":26})

Now generate the ENDF output (using force=True to avoid an error if rerunning the notebook).

In [ ]:
res = ncmat2endf.ncmat2endf(cfg_extn,endf_metadata=m, smin=5e-98, force=True)

In [ ]:
import pprint
pprint.pprint(res)

Now we can use a small utility script to invoke NJOY in order to create an ACE file from the ENDF file we just produced.

In [ ]:
from ncrystal_notebook_utilities.endf2ace import convert_endf_tsl_to_ace

endf_fn = 'tsl_Be.endf'
ace_fn = 'c_Be-293.6K.ace'
ace_fn, xsdir_fn = convert_endf_tsl_to_ace(endf_fn, ace_filename=ace_fn,ace_name='c_Be')

Use OpenMC to plot the data and compare against the NCrystal calculated cross-section.

In [ ]:
energy = np.geomspace(1e-3, 1.0,100000)

import openmc
thermal_data = openmc.data.ThermalScattering.from_ace('c_Be-293.6K.ace')
ace_elastic_xs = thermal_data.elastic.xs['294K']
ace_inelastic_xs = thermal_data.inelastic.xs['294K']
ace_total_xs = ace_elastic_xs(energy) + ace_inelastic_xs(energy)

plt.figure()
plt.plot(energy, ace_total_xs, 'b-', label='293.6 K NCrystal->ENDF->NJOY->ACE->OpenMC with extinction')
plt.plot(energy, mat_extn.scatter.xsect(energy), 'r-', label='293.6 K Direct NCrystal with extinction')
plt.plot(energy, mat_std.scatter.xsect(energy), 'y:', label='293.6 K Direct NCrystal without extinction')
plt.legend()
plt.xscale('log')
plt.xlabel('Energy [eV]')
plt.ylabel('Micro scattering cross section [b/atom]')
plt.show()


It is seen that the data from the ENDF file does to some degree account for the extinction, but that it can not reproduce the exact behavior between Bragg edges. This is because the ENDF format can only accomodate extinction models by "shoe-horning" them into standard Bragg edge shapes. More complicated extinction models, including those accounting for secondary extinction, are likely to fare even worse.